[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xylem-lab/GEO-ANOM-dashboard/blob/research/accuracy-assessment-2026-11-05/notebooks/14_accuracy_assessment.ipynb)

# 14 — Task 1 accuracy assessment by design

**Question.** How accurate is the frozen poultry-house map (release_v1: 2,833 houses, 6.98 km² roof), measured with a probability sample, a written protocol and two independent interpreters? Precision, recall, houses standing and total roof, each with a 95% confidence interval.

**Replaces** the single-reviewer 97% precision, the informal 21/24 still-standing check and the corrected roof 7.79 km² built on it. The objective numbers against the USGS 2016/17 tracing (recall 80.5%, precision 88.2%) stay as they are and are shown next to the new ones.

**Protocol:** [`docs/TASK1_ACCURACY_PROTOCOL.md`](../docs/TASK1_ACCURACY_PROTOCOL.md) (v1, frozen before the sample was drawn; its SHA-256 is in every saved file). Read it before labelling.

**How.**
1. Every map outline and every traced house goes in a stratum (S1 outline on a traced house, S2 outline not traced, S3 outline outside the traced area, S4 traced house the map misses), and the Eastern Shore is cut into 1 km cells (S6).
2. A stratified random sample by county is drawn **once** and saved with its SHA-256.
3. Two interpreters label every sampled item blind (NAIP 2023 photo, crosshair only, own random order): calibration (30) → pilot (75, kappa gate ≥ 0.8) → the rest → 400 cells.
4. Agreement (Cohen's kappa) before adjudication; disagreements adjudicated with a recorded reason.
5. Stratified estimators with finite-population-corrected bootstrap 95% CIs.

**Inputs.** `Task1_results/final/release_v1/` (frozen map, MANIFEST SHA-256), the USGS tracing (SHA-checked by `env.py`), county outlines, NAIP 2023 streamed from Planetary Computer (never stored).
**Outputs** (shared drive only: they name places). `Task1_results/accuracy_v1/`: `frame.geojson` (every item, its stratum, rank, role), `sample_meta.json`, `labels/labels_<interpreter>.csv`, `adjudicated.csv`, `estimates.csv`, `strata.csv`, `agreement.csv`, `run_meta.json`. Git gets only the anonymous summary in `docs/`.

**Time.** Steps 1–2: ~10 s. Labelling: ~8 h per interpreter (objects ~15 s each, cells ~40 s each; photos load in the background). Step 9: ~1 min.

**Interpreters:** set `INTERPRETER` below to your initials. With it empty, every labelling screen is skipped, so Run All always works.

In [ ]:
# ── Setup: the same cell works in Google Colab and locally ──────────────────────
import os, sys, json, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "research/accuracy-assessment-2026-11-05"   # branch, tag or commit to run; the exact commit is recorded in run_meta.json

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                     # results go to the shared drive's Task1_results
    ROOT = Path("/content/GEO-ANOM-dashboard")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/xylem-lab/GEO-ANOM-dashboard", str(ROOT)], check=True)
    # always bring the copy up to date (a reused Colab session keeps an old copy)
    head = lambda: subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout
    before = head()
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=True)
    remote = f"origin/{CODE_VERSION}"
    target = remote if subprocess.run(["git", "-C", str(ROOT), "rev-parse", "-q", "--verify", remote],
                                      capture_output=True).returncode == 0 else CODE_VERSION
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", target], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
    if head() != before and "geo_anom" in sys.modules:   # new code, but the old version is already loaded
        raise RuntimeError("Code updated. Runtime -> Restart session, then Run all again.")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
# reload edited geo_anom/ code automatically, without restarting the kernel
if "google.colab" not in sys.modules:   # Colab's IPython can't load autoreload on Python 3.12+ (no `imp`)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 100)

from geo_anom.task1 import env, region
from geo_anom.task1 import county as C
from geo_anom.task1 import accuracy as A
from geo_anom.task1.evaluate import load_ground_truth, GROUND_TRUTH
print("running in", "Colab" if IN_COLAB else "local", "| results ->", env.results_root())

## Settings

In [ ]:
INTERPRETER = ""                 # your initials, e.g. "UA" -- empty: labelling screens are skipped
INTERPRETERS = ["UA", ""]        # the two interpreters for agreement and adjudication (second: Jenny Ha or Pradeep)
SEED = 20261105                  # protocol v1
RELEASE = env.results_root() / "final" / "release_v1"
ACC = A.acc_dir()                # Task1_results/accuracy_v1 on the shared drive
LABELS_DIR = ACC / "labels"
PILOT_FAILED = False             # set True only if the pilot kappa is < 0.8 (step 4): pilot items are replaced
print("protocol", A.PROTOCOL.name, A.sha256_file(A.PROTOCOL)[:12], "| results ->", ACC)

## 1. Frames: every outline and traced house in its stratum, and the 1 km cells

Matching as in notebook 11 (`final.score`): an outline is *on* a traced house if they intersect; the traced extent of a county is the hull of its traced houses + 500 m. The counts must equal release_v1's own tables (2,450 / 327 / 56 / 594); the cell checks that.

In [ ]:
release = gpd.read_file(RELEASE / "maryland_poultry_houses.geojson")
traced = load_ground_truth()
objects, cells, K = A.frames(release, traced)
N = objects.groupby("stratum").size()
assert (N["S1"], N["S2"], N["S3"], N["S4"]) == (2450, 327, 56, 594), N   # = release_v1 ladder / miss_correction
assert N[["S1", "S2", "S3"]].sum() == len(release)
counts = pd.crosstab(pd.concat([objects.county, cells.county], ignore_index=True),
                     pd.concat([objects.stratum, cells.stratum], ignore_index=True),
                     margins=True, margins_name="total")
display(counts)
print("outline / traced area on S1 pairs (k, used for the roof of missed houses):", {c: round(v, 3) for c, v in K.items()})

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 8))
es = gpd.GeoSeries([region.county_boundary(n).to_crs(A.UTM).unary_union for n in C.EASTERN_SHORE], crs=A.UTM)
colors = {"S6A": "#d95f02", "S6B": "#fdb863", "S6C": "#eeeeee"}
for h, c in colors.items():
    cells[cells.stratum == h].plot(ax=axs[0], color=c, edgecolor="none")
es.boundary.plot(ax=axs[0], color="k", lw=0.5)
axs[0].set_title("1 km cells: A contains a known house, B within 1 km, C farther")
es.boundary.plot(ax=axs[1], color="k", lw=0.5)
pal = {"S1": "#1b9e77", "S2": "#7570b3", "S3": "#e7298a", "S4": "#d95f02"}
for h, c in pal.items():
    o = objects[(objects.stratum == h) & objects.county.isin(C.EASTERN_SHORE)]
    o.centroid.plot(ax=axs[1], color=c, markersize=2 if h == "S1" else 5, label=f"{h} {A.STRATUM_NAMES[h]} ({len(o):,})")
axs[1].legend(loc="lower left", fontsize=8, markerscale=3); axs[1].set_title("Object strata (Eastern Shore)")
for ax in axs: ax.set_axis_off()
plt.tight_layout(); plt.show()

## 2. Draw the sample, once

Within each stratum and county the items get a random order (seed 20261105): the first *n* are the sample, the next are the reserve (replacements if the pilot fails), the last 10 of S1, S2 and S4 are the calibration items. Allocation is proportional by county with at least 5 per county (S3: all). The file is saved with its SHA-256; once it exists this cell only loads it and checks that the protocol, release and tracing have not changed.

In [ ]:
FRAME = ACC / "frame.geojson"
META = ACC / "sample_meta.json"
if FRAME.exists():
    frame = gpd.read_file(FRAME).to_crs(A.UTM)
    sm = json.load(open(META))
    assert A.sha256_file(FRAME) == sm["frame_sha256"], "frame.geojson changed since it was drawn"
    now = A.meta(RELEASE, GROUND_TRUTH, SEED)
    for k in ("protocol_sha256", "release_manifest_sha256", "traced_sha256"):
        if now[k] != sm[k]:
            print(f"WARNING: {k} differs from when the sample was drawn -- check the protocol version / inputs")
    print("loaded the sample drawn", sm["made_at"])
else:
    frame = A.draw(objects, cells, seed=SEED)
    frame.to_crs(4326).to_file(FRAME, driver="GeoJSON")
    sm = A.meta(RELEASE, GROUND_TRUTH, SEED, {"frame_sha256": A.sha256_file(FRAME), "k": K,
                                              "frame_counts": A.frame_counts(objects, cells).to_dict("records")})
    json.dump(sm, open(META, "w"), indent=1, default=str)
    print("drew and saved the sample:", FRAME)
if PILOT_FAILED and not (frame.role == "discarded pilot").any():
    frame = A.replace_pilot(frame)
    frame.to_crs(4326).to_file(FRAME, driver="GeoJSON")
    sm.update({"frame_sha256": A.sha256_file(FRAME), "pilot_replaced": pd.Timestamp.utcnow().isoformat()})
    json.dump(sm, open(META, "w"), indent=1, default=str)
    print("pilot replaced from the reserve")
print("frame SHA-256", sm["frame_sha256"][:16], "| protocol", sm["protocol_sha256"][:12], "| commit", sm["git_commit"])

In [ ]:
s = frame[frame.role == "sample"]
alloc = pd.crosstab(s.county, s.stratum, margins=True, margins_name="total")
display(alloc)
roles = frame.pivot_table(index="stratum", columns="role", values="key", aggfunc="count", fill_value=0)
roles["pilot"] = frame[frame.pilot].groupby("stratum").size().reindex(roles.index).fillna(0).astype(int)
display(roles)

# expected 95% half-widths for a share p (with the finite-population correction)
def half(p, n, N): return 1.96 * np.sqrt(p * (1 - p) / n * (1 - n / N))
print(f"S2 at p=0.90: +/-{half(.9, 150, 327):.3f}   S4 at p=0.85: +/-{half(.85, 150, 594):.3f}, at 0.5: +/-{half(.5, 150, 594):.3f}")

## 3. Calibration (30 items, both interpreters, before the pilot)

These items can never enter the sample. Label them, then compare with the other interpreter below and discuss. If the protocol wording turns out unclear, write v2 of the protocol **before** the pilot.

Labels: **house** (standing poultry house, idle counts), **gone** (demolished, roofless, bare slab), **under construction**, **not a house**, **unsure**. Tick *low confidence* when you're not sure; if you checked Google Earth, put the image date in the note.

In [ ]:
if INTERPRETER:
    store = A.Store(LABELS_DIR, INTERPRETER)
    A.label_ui(A.queue(frame, INTERPRETER, "calibration", SEED), store, "calibration")
else:
    print("INTERPRETER is empty: labelling skipped")

In [ ]:
labels = A.load_labels(LABELS_DIR)
def compare(phase):
    L = labels[labels.phase == phase]
    if L.empty or L.reviewer.nunique() < 2:
        print(f"{phase}: needs both interpreters ({L.reviewer.nunique() if len(L) else 0} so far)"); return None
    w = L.pivot(index="key", columns="reviewer", values="value").dropna()
    w["stratum"] = frame.set_index("key").stratum.reindex(w.index).values
    return w
cal = compare("calibration")
if cal is not None:
    r = [c for c in cal.columns if c != "stratum"]
    print(f"calibration: same label on {(cal[r[0]] == cal[r[1]]).mean():.0%} of {len(cal)}")
    display(cal[cal[r[0]] != cal[r[1]]])

## 4. Pilot (75 items) and the kappa gate

25 random sampled items each from S1, S2 and S4. Both interpreters label them; then Cohen's kappa on house / not house.
- **κ ≥ 0.8** → go on to step 5 (pilot labels are kept).
- **κ < 0.8** → write protocol v2, set `PILOT_FAILED = True` in Settings and rerun from step 2: the pilot items are set aside and replaced from the reserve (same stratum and county). Never relabel the same items after seeing the disagreements.

In [ ]:
if INTERPRETER:
    A.label_ui(A.queue(frame, INTERPRETER, "pilot", SEED), store, "pilot")

In [ ]:
R2 = [r for r in INTERPRETERS if r]
pilot_keys = frame[frame.pilot & (frame.role == "sample")].key
if len(R2) == 2 and set(R2) <= set(labels.reviewer):
    pa = A.agreement(labels, frame, R2, keys=pilot_keys)
    display(pa)
    k_ = pa.set_index("items").loc["all objects", "kappa (house / not)"]
    n_ = pa.set_index("items").loc["all objects", "n"]
    print(f"pilot kappa {k_:.2f} on {n_} of {len(pilot_keys)} items ->",
          "PASS: go on" if (k_ >= 0.8 and n_ == len(pilot_keys)) else "not complete" if n_ < len(pilot_keys) else "FAIL: see above")
else:
    print("pilot gate: needs both interpreters' labels (set INTERPRETERS)")

## 5. Main labelling (the rest of the sampled objects, ~385 items)

In [ ]:
if INTERPRETER:
    A.label_ui(A.queue(frame, INTERPRETER, "main", SEED), store, "main")

## 6. Cells (400)

The dashed square is the 1 km cell; every building already known (map or tracing) has a cyan outline. Count the **standing poultry houses with their centre inside the square and no outline**. In the note, say where (e.g. "NE ninth, 3 houses"). Most cells will be 0.

In [ ]:
known = gpd.GeoDataFrame(geometry=pd.concat([objects[objects.stratum != "S3"].geometry,
                                               traced.to_crs(A.UTM).geometry], ignore_index=True), crs=A.UTM)
if INTERPRETER:
    A.cell_ui(A.queue(frame, INTERPRETER, "cells", SEED), store, known)

## 7. Agreement before adjudication

In [ ]:
labels = A.load_labels(LABELS_DIR)
done = labels.groupby(["reviewer", "phase"]).size().unstack(fill_value=0) if len(labels) else pd.DataFrame()
display(done)
sample_keys = frame[frame.role == "sample"].key
if len(R2) == 2 and set(R2) <= set(labels.reviewer):
    agree = A.agreement(labels, frame, R2, keys=sample_keys)
    display(agree)
    print(A.cell_agreement(labels, R2))
    agree.to_csv(ACC / "agreement.csv", index=False)
else:
    agree = None
    print("agreement: needs both interpreters")

## 8. Adjudication

Every key where the two differ, or either said unsure. Look at each together (the notes say who saw what), decide, and record it with `A.adjudicate(...)`: the decision, the reason and who decided. If you still disagree, the third person decides; if they can't, leave it (it stays *unsure*).

In [ ]:
ADJ = ACC / "adjudicated.csv"
adj = pd.read_csv(ADJ, dtype={"value": str}) if ADJ.exists() else None
if len(R2) == 2 and set(R2) <= set(labels.reviewer):
    q = A.adjudication_queue(labels[labels.key.isin(sample_keys)], R2)
    q = q.merge(frame[["key", "stratum", "county"]], on="key")
    left = q[~q.key.isin(adj.key)] if adj is not None else q
    print(f"{len(q)} to adjudicate, {len(left)} not yet decided")
    display(left.head(30))
# example (one call per decision):
# A.adjudicate(ADJ, "usgs:1234", "gone", "bare slab on the 2023 photo; Google Earth 2023-09 agrees", "UA+JH")

## 9. Estimates

Agreed label, else the adjudicated one, else *unsure*. Main estimate: *unsure* and *under construction* count as not a house; sensitivity rows count them as houses. Per-stratum shares come with Wilson and exact 95% intervals; the combined estimates with finite-population-corrected bootstrap intervals.

In [ ]:
if len(R2) == 2 and set(R2) <= set(labels.reviewer):
    final = A.final_labels(labels[labels.key.isin(sample_keys)], R2, adj)
    labelled = frame[(frame.role == "sample")].key.isin(final.index)
    print(f"sampled items with a final label: {labelled.sum()} of {len(labelled)}")
    strata = A.stratum_table(frame, final)
    display(strata)
    runs = {"main": {}, "unsure = house": {"unsure_as_house": True},
            "under construction = house": {"uc_as_house": True}}
    est = pd.concat({k: A.estimate(frame, final, sm["k"], **kw) for k, kw in runs.items()}, names=["run"])
    display(est.unstack("run").round(4))
    strata.to_csv(ACC / "strata.csv"); est.to_csv(ACC / "estimates.csv")
    json.dump({**A.meta(RELEASE, GROUND_TRUTH, SEED), "frame_sha256": sm["frame_sha256"], "interpreters": R2,
               "labelled": int(labelled.sum()), "notebook": "14_accuracy_assessment.ipynb"},
              open(ACC / "run_meta.json", "w"), indent=1, default=str)
    print("objective numbers vs the USGS tracing (unchanged): recall 0.805, precision 0.882")
else:
    print("estimates: need both interpreters' labels")

## What this shows

*Rewrite this cell from the outputs once labelling and adjudication are done.*

**State on 2026-11-05 (sample drawn, nothing labelled yet).**
- The frames reproduce release_v1 exactly: 2,450 outlines on a traced house (S1), 327 outlines in the traced area on no traced house (S2), 56 outlines outside the traced area (S3: 30 Eastern Shore, 26 west of the Bay), 594 traced houses the map misses (S4). The Eastern Shore has 9,361 1 km cells: 1,137 contain a known house, 2,748 are within 1 km of one, 5,476 are farther.
- The sample (protocol v1, seed 20261105) is saved on the shared drive with its SHA-256: 100 / 150 / 56 / 150 objects in S1–S4, 400 cells, 75 pilot items, 30 calibration items, the rest of each random order kept as reserve.
- No accuracy number comes from this notebook yet. Until it does, the only Task 1 accuracy numbers to quote are recall 80.5% and precision 88.2% against the USGS 2016/17 tracing (notebooks 11–12).